In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

import struct

sns.set_theme()

In [36]:
data_dir = "/home/patrick/Projects/DataScienceAndMachineLearning/Data/"

names = ["displacement", "mpg", "cylinders", "horsepower", "weight", "acceleration", "model_year", "origin", "car_name"]
df = pd.read_table(data_dir + 'auto-mpg.data', sep = r'\s+', names=names)
df = df.drop(columns=["car_name"])

df["is_USA"] = (df["origin"] == 1).astype(np.int64)
df["is_Europe"] = (df["origin"] == 2).astype(np.int64)
df["is_Japan"] = (df["origin"] == 3).astype(np.int64)

df["horsepower"] = df["horsepower"].replace("?", np.nan).astype(np.float64)
meanval = df["horsepower"].mean()
df["horsepower"] = df["horsepower"].fillna(meanval)
df = df.drop(columns=["origin", "model_year"])
df = df.astype(np.float64)
# print(np.where(df == "?"))
df

,displacement,mpg,cylinders,horsepower,weight,acceleration,is_USA,is_Europe,is_Japan
0,18.0,8.0,307.0,130.0,3504.0,12.0,1.0,0.0,0.0
1,15.0,8.0,350.0,165.0,3693.0,11.5,1.0,0.0,0.0
2,18.0,8.0,318.0,150.0,3436.0,11.0,1.0,0.0,0.0
3,16.0,8.0,304.0,150.0,3433.0,12.0,1.0,0.0,0.0
4,17.0,8.0,302.0,140.0,3449.0,10.5,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
393,27.0,4.0,140.0,86.0,2790.0,15.6,1.0,0.0,0.0
394,44.0,4.0,97.0,52.0,2130.0,24.6,0.0,1.0,0.0
395,32.0,4.0,135.0,84.0,2295.0,11.6,1.0,0.0,0.0
396,28.0,4.0,120.0,79.0,2625.0,18.6,1.0,0.0,0.0


In [37]:
# target values
df['mpg'].value_counts()

mpg
4.0    204
8.0    103
6.0     84
3.0      4
5.0      3
Name: count, dtype: int64

In [38]:
X_data = df.drop(columns=["mpg"]).to_numpy()
y_data = df["mpg"].to_numpy()
X_data

array([[ 18., 307., 130., ...,   1.,   0.,   0.],
       [ 15., 350., 165., ...,   1.,   0.,   0.],
       [ 18., 318., 150., ...,   1.,   0.,   0.],
       ...,
       [ 32., 135.,  84., ...,   1.,   0.,   0.],
       [ 28., 120.,  79., ...,   1.,   0.,   0.],
       [ 31., 119.,  82., ...,   1.,   0.,   0.]], shape=(398, 8))

In [39]:
def euclidean_distance(p, q):
    return np.sqrt((p - q) @ (p - q))

In [40]:
train_cutoff = round(0.7 * X_data.shape[0])
train_X = X_data[:train_cutoff, :]
test_X = X_data[train_cutoff:, :]
train_y = y_data[:train_cutoff]
test_y = y_data[train_cutoff:]
train_X

array([[ 18. , 307. , 130. , ...,   1. ,   0. ,   0. ],
       [ 15. , 350. , 165. , ...,   1. ,   0. ,   0. ],
       [ 18. , 318. , 150. , ...,   1. ,   0. ,   0. ],
       ...,
       [ 21.6, 121. , 115. , ...,   0. ,   1. ,   0. ],
       [ 16.2, 163. , 133. , ...,   0. ,   1. ,   0. ],
       [ 31.5,  89. ,  71. , ...,   0. ,   1. ,   0. ]], shape=(279, 8))

In [41]:
def get_k_nearest_neighbors(dist_func, train_X, train_y, point, k):
    # build distance dictionary
    distances = []
    for i in range(np.shape(train_X)[0]):
        dist = dist_func(train_X[i], point)
        distances.append([train_X[i], train_y[i], dist])
    return sorted(distances, key=lambda x: x[-1])[:k]

def predict_k_nearest_neighbors(dist_func, train_x, train_y, point, k):
    nearest_neighbors = get_k_nearest_neighbors(dist_func, train_x, train_y, point, k)
    prediction = 0.0
    for i in range(k):
        neighbor = nearest_neighbors[i]
        neighbor_prediction = neighbor[1]
        prediction += neighbor_prediction / k
    return round(prediction)

In [42]:
def test_k_nearest_neighbors(dist_func, train_X, train_y, test_X, test_y, k):
    good_preds = 0
    for i in range(test_X.shape[0]):
        pred = predict_k_nearest_neighbors(dist_func, train_X, train_y, test_X[i, :], k)
        good_preds += 1 if pred == test_y[i] else 0
    good_preds /= test_X.shape[0]
    return good_preds

In [43]:
print("  k | Successful prediction rate on test data with Euclidean distance")
print("----|----------------------------------------------------------------")
for k in range(1, 21):
    pred_rate = test_k_nearest_neighbors(euclidean_distance, train_X, train_y, test_X, test_y, k)
    print(f" {k:2d} | {pred_rate:1.3f}")

  k | Successful prediction rate on test data with Euclidean distance
----|----------------------------------------------------------------
  1 | 0.882
  2 | 0.849
  3 | 0.798
  4 | 0.840
  5 | 0.824
  6 | 0.815
  7 | 0.798
  8 | 0.815
  9 | 0.798
 10 | 0.790
 11 | 0.790
 12 | 0.790
 13 | 0.790
 14 | 0.765
 15 | 0.756
 16 | 0.765
 17 | 0.748
 18 | 0.739
 19 | 0.739
 20 | 0.739
